In [10]:
# ==================================================
# Module 1. Import Packages and Global Settings
# ==================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


# Suppress repeated numerical warnings from fixed-effect regressions
warnings.filterwarnings("ignore")


# File paths
BASE_DIR = Path(r"C:\Users\linjiajun\Desktop\5810data")

DATA_FILE = BASE_DIR / "全部游戏数据汇总.xlsx"
ACCIDENT_FILE = BASE_DIR / "Accident.xlsx"


# Event windows
WINDOWS = [3, 7, 14, 30, 90]


# Outcome variables for regressions
OUTCOME_VARS = [
    "ln_Revenue",
    "ln_DAU",
    "ln_Downloads",
    "RPD ($)",
    "ARPDAU ($)"
]


# Name mapping between accident table and operation data
NAME_MAP = {
    "崩坏：星穹铁道": "崩坏星穹铁道",
    "重返未来：1999": "重返未来1999",
    "少女前线2：追放": "少女前线2",
    "火影忍者手游": "火影忍者"
}


# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

In [11]:
# ==================================================
# Module 2. Load and Clean Raw Data
# ==================================================

def load_data(data_file, accident_file):
    """
    Load operation data and accident event data.
    """

    data = pd.read_excel(data_file)
    accident = pd.read_excel(accident_file)

    # Clean column names
    data.columns = data.columns.str.strip()
    accident.columns = accident.columns.str.strip()

    # Parse dates
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
    accident["Date"] = pd.to_datetime(accident["Date"], errors="coerce")

    # Standardize game names
    accident["游戏名"] = accident["游戏名"].replace(NAME_MAP)

    # Remove invalid values
    data = data.replace([np.inf, -np.inf], np.nan)
    accident = accident.dropna(subset=["Date"]).copy()

    return data, accident


data, accident = load_data(DATA_FILE, ACCIDENT_FILE)

print("=" * 100)
print("Operation data preview")
print(data.head())

print("\nAccident data preview")
print(accident.head())

print("\nOperation data columns")
print(data.columns.tolist())

print("\nAccident data columns")
print(accident.columns.tolist())

Operation data preview
  Folder Game Name   App Name      App ID       Date Country / Region   Platform          DAU  Downloads  Revenue ($)  RPD ($)  ARPDAU ($)
0             明日方舟  Arknights  1464872022 2020-01-16               US  App Store  7918.000000       8678 39457.690000 4.546864    4.983290
1             明日方舟  Arknights  1464872022 2020-01-17               US  App Store 12419.000000       8048 45963.620000 5.711185    3.701073
2             明日方舟  Arknights  1464872022 2020-01-18               US  App Store 15771.000000       7479 43153.660000 5.769977    2.736267
3             明日方舟  Arknights  1464872022 2020-01-19               US  App Store 18539.000000       7261 42298.050000 5.825375    2.281571
4             明日方舟  Arknights  1464872022 2020-01-20               US  App Store 19581.000000       5658 32530.540000 5.749477    1.661332

Accident data preview
  游戏名       Date 事故分类
0  原神 2020-12-01   A类
1  原神 2022-02-09   A类
2  原神 2022-03-30   A类
3  原神 2022-04-04   A类
4  原神 2022

In [12]:
# ==================================================
# Module 3. Create Outcome Variables
# ==================================================

def add_log_variables(df):
    """
    Create log-transformed outcome variables.
    """

    df = df.copy()

    df["ln_Revenue"] = np.log1p(df["Revenue ($)"])
    df["ln_DAU"] = np.log1p(df["DAU"])
    df["ln_Downloads"] = np.log1p(df["Downloads"])

    return df

In [13]:
# ==================================================
# Module 4. Construct Event Windows
# ==================================================

def make_event_window(data, accident, window_days, accident_types):
    """
    Build event-window data for selected accident types.
    """

    selected_events = accident[accident["事故分类"].isin(accident_types)].copy()
    selected_events = selected_events.reset_index(drop=True)
    selected_events["event_id"] = selected_events.index + 1

    window_list = []
    no_match_list = []

    for _, row in selected_events.iterrows():
        event_id = row["event_id"]
        game_name = row["游戏名"]
        event_date = row["Date"]
        accident_type = row["事故分类"]

        start_date = event_date - pd.Timedelta(days=window_days)
        end_date = event_date + pd.Timedelta(days=window_days)

        temp = data[
            (data["Folder Game Name"] == game_name) &
            (data["Date"] >= start_date) &
            (data["Date"] <= end_date)
        ].copy()

        if temp.empty:
            no_match_list.append({
                "event_id": event_id,
                "game_name": game_name,
                "event_date": event_date,
                "accident_type": accident_type
            })
            continue

        temp["event_id"] = event_id
        temp["event_game"] = game_name
        temp["event_date"] = event_date
        temp["accident_type"] = accident_type
        temp["event_time"] = (temp["Date"] - event_date).dt.days
        temp["post"] = np.where(temp["event_time"] >= 0, 1, 0)

        window_list.append(temp)

    if not window_list:
        return pd.DataFrame(), pd.DataFrame(no_match_list)

    event_window = pd.concat(window_list, ignore_index=True)
    event_window = event_window.replace([np.inf, -np.inf], np.nan)
    event_window = add_log_variables(event_window)

    no_match = pd.DataFrame(no_match_list)

    return event_window, no_match

In [14]:
# ==================================================
# Module 5. Before-After Descriptive Analysis
# ==================================================

def before_after_summary(event_window, window_days):
    """
    Compare average outcomes before and after accident dates.
    """

    df = event_window.copy()

    df["period"] = pd.Series(index=df.index, dtype="object")

    df.loc[
        (df["event_time"] >= -window_days) &
        (df["event_time"] <= -1),
        "period"
    ] = "Before"

    df.loc[
        (df["event_time"] >= 0) &
        (df["event_time"] <= window_days),
        "period"
    ] = "After"

    df = df.dropna(subset=["period"]).copy()

    variables = [
        "Revenue ($)",
        "DAU",
        "Downloads",
        "RPD ($)",
        "ARPDAU ($)",
        "ln_Revenue",
        "ln_DAU",
        "ln_Downloads"
    ]

    result_list = []

    for var in variables:
        temp = (
            df.groupby(
                [
                    "event_id",
                    "event_game",
                    "event_date",
                    "accident_type",
                    "Country / Region",
                    "period"
                ],
                as_index=False
            )[var]
            .mean()
        )

        pivot = temp.pivot_table(
            index=[
                "event_id",
                "event_game",
                "event_date",
                "accident_type",
                "Country / Region"
            ],
            columns="period",
            values=var
        ).reset_index()

        if "Before" not in pivot.columns or "After" not in pivot.columns:
            continue

        pivot["variable"] = var
        pivot["change"] = pivot["After"] - pivot["Before"]

        # Avoid division by zero
        pivot["change_rate"] = np.where(
            pivot["Before"] > 0,
            pivot["change"] / pivot["Before"],
            np.nan
        )

        result_list.append(pivot)

    if not result_list:
        return pd.DataFrame(), pd.DataFrame()

    detail = pd.concat(result_list, ignore_index=True)
    detail = detail.replace([np.inf, -np.inf], np.nan)

    summary = (
        detail
        .groupby(["accident_type", "variable"], as_index=False)
        .agg({
            "Before": "mean",
            "After": "mean",
            "change": "mean",
            "change_rate": "mean"
        })
    )

    return detail, summary

In [15]:
# ==================================================
# Module 6. Fixed Effects Regression Setup
# ==================================================

def get_fe_formula(window_days):
    """
    Select fixed effects based on event-window length.
    """

    if window_days <= 3:
        return "C(game_country_fe)", "Game-country FE only"

    if window_days >= 90:
        return "C(game_country_fe) + C(time_fe)", "Game-country FE + month FE"

    return "C(game_country_fe) + C(time_fe)", "Game-country FE + date FE"


def prepare_regression_data(event_window, window_days, region=None):
    """
    Prepare event-window data for fixed-effect regression.
    """

    df = event_window.copy()

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["event_time"] = pd.to_numeric(df["event_time"], errors="coerce")
    df["post"] = pd.to_numeric(df["post"], errors="coerce")

    df = df[
        (df["event_time"] >= -window_days) &
        (df["event_time"] <= window_days)
    ].copy()

    if region is not None:
        df = df[df["Country / Region"] == region].copy()

    df = df.replace([np.inf, -np.inf], np.nan)

    df["game_country_fe"] = (
        df["event_game"].astype(str) +
        "_" +
        df["Country / Region"].astype(str)
    )

    if window_days >= 90:
        df["time_fe"] = df["Date"].dt.strftime("%Y-%m")
    elif window_days > 3:
        df["time_fe"] = df["Date"].dt.strftime("%Y-%m-%d")
    else:
        df["time_fe"] = "no_time_fe"

    return df

In [19]:
# ==================================================
# Module 8. Batch Analysis by Accident Type and Window
# ==================================================

def run_window_analysis(data, accident, accident_types, sample_name, windows):
    """
    Run event-window construction, descriptive analysis,
    and regressions for one accident sample.
    """

    all_regression_results = []
    all_summary_results = []
    all_no_match = []

    for window_days in windows:
        print("=" * 100)
        print(f"Sample: {sample_name} | Window: [-{window_days}, +{window_days}]")

        event_window, no_match = make_event_window(
            data=data,
            accident=accident,
            window_days=window_days,
            accident_types=accident_types
        )

        if event_window.empty:
            print("No event-window data generated.")
            continue

        print("Event-window rows:", len(event_window))
        print("Matched events:", event_window["event_id"].nunique())

        if not no_match.empty:
            no_match["sample"] = sample_name
            no_match["window"] = f"[-{window_days}, +{window_days}]"
            all_no_match.append(no_match)

            print("Unmatched events:")
            print(no_match)

        _, summary = before_after_summary(event_window, window_days)

        if not summary.empty:
            summary["sample"] = sample_name
            summary["window"] = f"[-{window_days}, +{window_days}]"
            all_summary_results.append(summary)

        reg_result = run_regression(
            event_window=event_window,
            window_days=window_days,
            sample_name=sample_name
        )

        all_regression_results.append(reg_result)

        print(reg_result.to_string(index=False))

    regression_table = (
        pd.concat(all_regression_results, ignore_index=True)
        if all_regression_results else pd.DataFrame()
    )

    summary_table = (
        pd.concat(all_summary_results, ignore_index=True)
        if all_summary_results else pd.DataFrame()
    )

    no_match_table = (
        pd.concat(all_no_match, ignore_index=True)
        if all_no_match else pd.DataFrame()
    )

    return regression_table, summary_table, no_match_table

In [20]:
# ==================================================
# Module 9. Regional Heterogeneity Analysis
# ==================================================

def run_region_analysis(data, accident, accident_types, sample_name, window_days, regions):
    """
    Run region-specific regressions for selected accident types.
    """

    event_window, no_match = make_event_window(
        data=data,
        accident=accident,
        window_days=window_days,
        accident_types=accident_types
    )

    if event_window.empty:
        return pd.DataFrame(), no_match

    region_results = []

    for region in regions:
        result = run_regression(
            event_window=event_window,
            window_days=window_days,
            sample_name=sample_name,
            region=region
        )

        region_results.append(result)

    region_table = pd.concat(region_results, ignore_index=True)

    return region_table, no_match

In [21]:
# ==================================================
# Module 10. Main Execution and Output
# ==================================================

# Main sample: A + B
reg_ab, summary_ab, no_match_ab = run_window_analysis(
    data=data,
    accident=accident,
    accident_types=["A类", "B类"],
    sample_name="A+B",
    windows=WINDOWS
)


# A-only sample
reg_a, summary_a, no_match_a = run_window_analysis(
    data=data,
    accident=accident,
    accident_types=["A类"],
    sample_name="A-only",
    windows=WINDOWS
)


# B-only sample
reg_b, summary_b, no_match_b = run_window_analysis(
    data=data,
    accident=accident,
    accident_types=["B类"],
    sample_name="B-only",
    windows=WINDOWS
)


# Regional heterogeneity for the main 14-day window
region_result, region_no_match = run_region_analysis(
    data=data,
    accident=accident,
    accident_types=["A类", "B类"],
    sample_name="A+B",
    window_days=14,
    regions=["CN", "US", "JP"]
)


# Combine all results
all_regression_results = pd.concat(
    [reg_ab, reg_a, reg_b, region_result],
    ignore_index=True
)

all_summary_results = pd.concat(
    [summary_ab, summary_a, summary_b],
    ignore_index=True
)

all_no_match = pd.concat(
    [no_match_ab, no_match_a, no_match_b],
    ignore_index=True
)


print("=" * 100)
print("Final Regression Results")
print(all_regression_results.to_string(index=False))

print("=" * 100)
print("Before-After Summary")
print(all_summary_results.to_string(index=False))

print("=" * 100)
print("Unmatched Events")
print(all_no_match.to_string(index=False))

Sample: A+B | Window: [-3, +3]
Event-window rows: 837
Matched events: 41
Unmatched events:
   event_id game_name event_date accident_type sample    window
0        27      少女前线 2024-12-06            A类    A+B  [-3, +3]
1        28      少女前线 2025-06-12            A类    A+B  [-3, +3]
2        29      少女前线 2025-06-13            A类    A+B  [-3, +3]
sample region   window      outcome  post_coef  std_error  p_value sig  percent_effect  n_obs  n_events  r_squared        fixed_effects note
   A+B    ALL [-3, +3]   ln_Revenue   0.513379   0.218831 0.018976  **        0.670928    837        41   0.608814 Game-country FE only     
   A+B    ALL [-3, +3]       ln_DAU   0.013034   0.024124 0.589015            0.013119    788        41   0.886319 Game-country FE only     
   A+B    ALL [-3, +3] ln_Downloads   0.147969   0.066057 0.025090  **        0.159477    837        41   0.751909 Game-country FE only     
   A+B    ALL [-3, +3]      RPD ($)  14.251815   7.223718 0.048505  **             NaN   